In [196]:
import pandas as pd
import numpy as np
import random

In [197]:
school = 'school_1'

# Define paths
raw_data_folder = '../data/raw_data'
processed_data_folder = '../data/processed_data'
processed_data_folder_school = f'{processed_data_folder}/{school}'
print(f'Processed data folder: {processed_data_folder_school}')

Processed data folder: ../data/processed_data/school_1


In [198]:
# Read in processed data
constraints_students = pd.read_csv(f'{processed_data_folder_school}/constraints_students.csv')
constraints_teachers = pd.read_csv(f'{processed_data_folder_school}/constraints_teachers.csv')
current_groups = pd.read_csv(f'{processed_data_folder_school}/current_groups.csv')
group_preferences = pd.read_csv(f'{processed_data_folder_school}/group_preferences.csv')
info_students = pd.read_csv(f'{processed_data_folder_school}/info_students.csv')
info_teachers = pd.read_csv(f'{processed_data_folder_school}/info_teachers.csv')
group_preferences

n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2 = group_preferences.iloc[0]


In [199]:
# Set seed for reproducibility
seed = 42

In [207]:
# def violates_min_group_size(group, min_group_size):
#     return len(group) < min_group_size

def violates_student_constraints(sid, gid, assignment, pairs_df):
    for _, row in pairs_df.iterrows():
        s1, s2, together = row['Student 1'], row['Student 2'], row['Together']
        if together == 'Yes':
            if sid == s1:
                for g, members in assignment.items():
                    if s2 in members and g != gid:
                        return True
            elif sid == s2:
                for g, members in assignment.items():
                    if s1 in members and g != gid:
                        return True
        elif together == 'No':
            if sid == s1 and s2 in assignment[gid]:
                return True
            elif sid == s2 and s1 in assignment[gid]:
                return True
    return False

def violates_student_pairs(student, groups, constraints_students):
    for _, (s1, s2, together) in constraints_students.iterrows():
        # Check if the student is in the pair
        if student == s1 or student == s2:
            # Check if both students have been assigned to a group
            # Get group of s1
            group_s1 = groups.get(s1)
            # Get group of s2
            group_s2 = groups.get(s2)

            # Check if both exist
            if group_s1 is not None and group_s2 is not None:
                # Check if violated if pair is required
                if together == 'Yes' and group_s1 != group_s2:
                        return True
                # Check if violated if pair is illegal
                elif together == 'No' and group_s1 == group_s2:
                        return True

                return False
            return True





In [ ]:

# Main assignment logic
def assign_students():
    for _ in range(1000):  # try multiple random attempts
        groups = {i: [] for i in range(num_groups)}
        care_counts = {i: {'Extra Care': 0, 'Extra Care 2': 0} for i in range(num_groups)}
        random.shuffle(students)
        success = True

        for student in students:
            assigned = False
            group_ids = list(groups.keys())
            random.shuffle(group_ids)
            for gid in group_ids:
                if len(groups[gid]) >= 10:
                    continue
                if student['extra_care'] == 'Yes' and care_counts[gid]['Extra Care'] >= max_extra_care:
                    continue
                if student['extra_care2'] == 'Yes' and care_counts[gid]['Extra Care 2'] >= max_extra_care2:
                    continue
                # Check student-student constraints
                if violates_together_constraints(student['id'], gid, groups, must_be_together, must_not_be_together):
                    continue
                # Check teacher constraints if relevant
                if violates_teacher_constraints(student['id'], gid, teacher_constraints):
                    continue

                groups[gid].append(student['id'])
                student['group'] = gid
                if student['extra_care'] == 'Yes':
                    care_counts[gid]['Extra Care'] += 1
                if student['extra_care2'] == 'Yes':
                    care_counts[gid]['Extra Care 2'] += 1
                assigned = True
                break

            if not assigned:
                success = False
                break

        if success and all(len(g) >= min_group_size for g in groups.values()):
            return groups  # Valid assignment found

    return None  # Couldn’t find valid assignment in attempts


# Helper functions like:
def violates_together_constraints(student_id, gid, groups, must_together, must_not_together):
    for (s1, s2) in must_together:
        if student_id == s1:
            for group_id, members in groups.items():
                if s2 in members:
                    return group_id != gid
        if student_id == s2:
            for group_id, members in groups.items():
                if s1 in members:
                    return group_id != gid
    for (s1, s2) in must_not_together:
        if student_id == s1:
            if s2 in groups[gid]:
                return True
        if student_id == s2:
            if s1 in groups[gid]:
                return True
    return False

def violates_teacher_constraints(student_id, gid, teacher_constraints):
    # Optional: Only if groups are mapped to teachers
    return False

In [200]:
# def build_student_clusters(groups, constraints_students, seed):
#     # Initialize clusters
#     random.seed(seed)
#     print(groups)
#     clusters = {i: [] for i in range(1, n_groups + 1)}

#     # Add students to clusters based on constraints
#     for _, row in constraints_students.iterrows():
#         student_1 = row['Student 1']
#         student_2 = row['Student 2']
#         together = row['Together']
#         print(f'Adding students {student_1} and {student_2} to clusters with together={together}')

#         if together == 'Yes':
#             # Add both students to the same group
#             group = random.choice(list(groups.keys()))
#             print(f'Adding students {student_1} and {student_2} to group {group}')
#             clusters[group].append(student_1)
#             clusters[group].append(student_2)
#         else:
#             # Add each student to a different group
#             group_1, group_2 = random.sample(list(groups.keys()), 2)
#             print(f'Adding student {student_1} to group {group_1} and student {student_2} to group {group_2}')
#             clusters[group_1].append(student_1)
#             clusters[group_2].append(student_2)

#     print(f'Clusters after student constraints: {clusters}')
#     return clusters

# build_student_clusters(groups, constraints_students, seed)

In [201]:


# list_constraint_students(constraints_students, constraints_teachers)

In [202]:
# groups = {}
# for i, teacher in enumerate(info_teachers['Teacher']):
#     groups[i + 1] = {'Teacher': teacher, 'Students': []}
# print(f'Groups: {groups}')



teachers_exclude = constraints_teachers[constraints_teachers['Together'] == 'No']
teachers_include = constraints_teachers[constraints_teachers['Together'] == 'Yes']
students_include = constraints_students[constraints_students['Together'] == 'Yes']
students_exclude = constraints_students[constraints_students['Together'] == 'No']

In [ ]:
def random_group(seed, available_groups):
    random.seed(seed)
    return random.choice(available_groups)


def valid_groups_max(groups, student, info_students, binary_col, max_allowed):
    valid_groups = []
    student_flag = info_students.loc[info_students['Student'] == student, binary_col].values[0]

    for group in groups:
        if student_flag == 1:
            count_flagged = sum(info_students.loc[info_students['Student'] == s, binary_col].values[0]
                for s in groups[group])
            if count_flagged < max_allowed:
                valid_groups.append(group)
        else:
            valid_groups.append(group)

    print(f'Valid groups for student {student} with {binary_col} = {student_flag}: {valid_groups}')
    return valid_groups


def get_sorted_students(groups):
    return [student for group in groups.values() for student in group]


def list_constraint_students(constraints_students, constraints_teachers):
    # Create set of all students mentioned in constraints students and teachers
    students_in_constraints = set(constraints_students['Student 1']).union(set(constraints_students['Student 2']))
    students_in_constraints.update(set(constraints_teachers['Student']))
    return students_in_constraints


def check_group_size(groups, min_group_size):
    for group in groups.values():
        if len(group) < min_group_size:
            return False
    return True


def check_max_extra_care(groups, info_students, max_extra_care_1):
    for group, students in groups.items():
        count = 0
        for student in students:
            care = info_students.loc[info_students['Student'] == student, 'Extra Care']
            if not care.empty and care.values[0] == 'Yes':
                count += 1
        print(f'Group {group} has {count} students needing extra care.')
        if count > max_extra_care_1:
            return False  # Limit exceeded for this group
    return True

In [204]:
def teacher_constraints(groups, constraints_teachers):
    for _, (student, teacher, together) in constraints_teachers.iterrows():
        if together == 'Yes':
            groups[teacher].append(student)
        else:
            available_groups = list(groups.keys())
            available_groups.remove(teacher)
            groups[random_group(seed, available_groups)].append(student)

    return groups

def get_student_group(students, sorted_students, groups):
    for student in students:
        if student in sorted_students:
            group = [group for group in groups if student in groups[group]]
            return group[0], student

    return None, None
    # return random_group(seed, list(groups.keys()))


def required_pairs(groups):
    students_include = constraints_students[constraints_students['Together'] == 'Yes']
    for _, (student_1, student_2, _) in students_include.iterrows():
        # If a student is already in a group, add the other student to the same group
        sorted_students = get_sorted_students(groups)
        student_group, student = get_student_group([student_1, student_2], sorted_students, groups)

        if student_group is None:
            # If neither student is in a group, assign both to a new random group
            student_group = random_group(seed, list(groups.keys()))
            groups[student_group].append(student_1)
            groups[student_group].append(student_2)
        else:
            # If one student is in a group, add the other student to the same group
            if student == student_1:
                groups[student_group].append(student_2)
            else:
                groups[student_group].append(student_1)

    return groups

def excluded_pairs(groups):
    students_exclude = constraints_students[constraints_students['Together'] == 'No']
    for _, (student_1, student_2, _) in students_exclude.iterrows():
        # If a student is already in a group, add the other student to another group
        sorted_students = get_sorted_students(groups)
        student_group, student = get_student_group([student_1, student_2], sorted_students, groups)

        if student_group is None:
            # If neither student is in a group, assign both to a different random group
            student_group = random_group(seed, list(groups.keys()))
            groups[student_group].append(student_1)
            available_groups = list(groups.keys())
            available_groups.remove(student_group)
            groups[random_group(seed, available_groups)].append(student_2)
        else:
            # If one student is in a group, add the other student to another group
            available_groups = list(groups.keys())
            available_groups.remove(student_group)
            student_group = random_group(seed, available_groups)

            if student == student_1:
                groups[student_group].append(student_2)
            else:
                groups[student_group].append(student_1)

    return groups

def student_constraints(groups, constraints_students):
    # Apply required pairs first
    groups = required_pairs(groups)
    print(f'Groups after required pairs: {groups}')

    # Then apply excluded pairs
    groups = excluded_pairs(groups)
    print(f'Groups after excluded pairs: {groups}')

    # Finish with adding remaining students
    remaining_students = [s for s in info_students['Student'] if s not in get_sorted_students(groups)]

    for student in remaining_students:
        correct_group_size = check_group_size(groups, min_group_size)
        if correct_group_size:
            # add remaining students to random groups
            group = random_group(seed, list(groups.keys()))
        else:
            # Add students to smallest group
            group = min(groups, key=lambda x: len(groups[x]))

        groups[group].append(student)

    return groups


def random_grouping(constraints_students, constraints_teachers):
    # Initialize groups
    groups = {}
    for _, teacher in enumerate(info_teachers['Teacher']):
        groups[teacher] = []

    # Apply teacher constraints
    groups = teacher_constraints(groups, constraints_teachers)
    print(f'Groups after teacher constraints: {groups}')
    print(f'Extra care in groups: {check_max_extra_care(groups, info_students, max_extra_care_1)}')

    # Apply student constraints
    groups = student_constraints(groups, constraints_students)
    print(f'Groups after student constraints: {groups}')

    # Check other constraints
    # Max extra care
    g = check_max_extra_care(groups, info_students, max_extra_care_1)
    print(f'Groups after max extra care check: {g}')

    return groups


random_grouping(constraints_students, constraints_teachers)

Groups after teacher constraints: {'T_01': ['S_02', 'S_26', 'S_28'], 'T_02': ['S_21'], 'T_03': ['S_10', 'S_02', 'S_30']}
Group T_01 has 1 students needing extra care.
Group T_02 has 1 students needing extra care.
Group T_03 has 0 students needing extra care.
Extra care in groups: True
Groups after required pairs: {'T_01': ['S_02', 'S_26', 'S_28'], 'T_02': ['S_21', 'S_08'], 'T_03': ['S_10', 'S_02', 'S_30', 'S_29', 'S_13', 'S_03', 'S_01']}
Groups after excluded pairs: {'T_01': ['S_02', 'S_26', 'S_28', 'S_07', 'S_09'], 'T_02': ['S_21', 'S_08', 'S_26'], 'T_03': ['S_10', 'S_02', 'S_30', 'S_29', 'S_13', 'S_03', 'S_01', 'S_23']}
Groups after student constraints: {'T_01': ['S_02', 'S_26', 'S_28', 'S_07', 'S_09', 'S_06', 'S_12'], 'T_02': ['S_21', 'S_08', 'S_26', 'S_04', 'S_05', 'S_11', 'S_14'], 'T_03': ['S_10', 'S_02', 'S_30', 'S_29', 'S_13', 'S_03', 'S_01', 'S_23', 'S_15', 'S_16', 'S_17', 'S_18', 'S_19', 'S_20', 'S_22', 'S_24', 'S_25', 'S_27']}
Group T_01 has 2 students needing extra care.
Gro

{'T_01': ['S_02', 'S_26', 'S_28', 'S_07', 'S_09', 'S_06', 'S_12'],
 'T_02': ['S_21', 'S_08', 'S_26', 'S_04', 'S_05', 'S_11', 'S_14'],
 'T_03': ['S_10',
  'S_02',
  'S_30',
  'S_29',
  'S_13',
  'S_03',
  'S_01',
  'S_23',
  'S_15',
  'S_16',
  'S_17',
  'S_18',
  'S_19',
  'S_20',
  'S_22',
  'S_24',
  'S_25',
  'S_27']}

In [205]:


# print(teachers_include['Student'].values)
# students_include['Teacher'] = None
# for _, (student_1, student_2, together, teacher) in students_include.iterrows():
#     print("student 1", student_1)
#     print("student 2", student_2)
#     if student_1 in teachers_include['Student'].values:
#         print("student_1 in teachers_include")
#         students_include.loc[students_include['Student 1'] == student_1, 'Teacher'] = teachers_include.loc[teachers_include['Student'] == student_1, 'Teacher'].values[0]
#     elif student_2 in teachers_include['Student'].values:
#         print("student_2 in teachers_include")
#         students_include.loc[students_include['Student 2'] == student_2, 'Teacher'] = teachers_include.loc[teachers_include['Student'] == student_2, 'Teacher'].values[0]

# print(students_include)

In [206]:
groups = {}
for i, teacher in enumerate(info_teachers['Teacher']):
    groups[i + 1] = {'Teacher': teacher, 'Students': []}
print(f'Groups: {groups}')

def student_pairs(students_include):
    student_pairs = []
    for _, (student_1, student_2, _) in students_include.iterrows():

        # Check if the students are already in a group
        added = False
        for pair in student_pairs:
            if student_1 in pair or student_2 in pair:
                # Add the students to the existing group
                pair.extend([student_1, student_2])
                added = True

        if not added:
            student_pairs.append([student_1, student_2])

    return [set(pair) for pair in student_pairs]

def group_pairs(pairs, teachers_include, teachers_exclude):
    # Loop through student pairs and assign them to groups
    for pair in pairs:
        # print(f'Processing pair: {pair}')
        added = False
        # Check if student in pair is paired to teacher
        for student in pair:
            if student in teachers_include['Student'].values:
                teacher = teachers_include.loc[constraints_teachers['Student'] == student, 'Teacher'].values[0]
                group = [group for group, info in groups.items() if info['Teacher'] == teacher]
                # print(f'Added pair to group {group[0]} with teacher {teacher}')

                # Add pair to the group
                groups[group[0]]['Students'].extend(pair)
                added = True

            # Check if student in pair is excluded from teacher
            elif student in teachers_exclude['Student'].values:
                teacher = teachers_exclude.loc[constraints_teachers['Student'] == student, 'Teacher'].values[0]
                group = [group for group, info in groups.items() if info['Teacher'] == teacher]

                # Add pair to random group that is not the excluded group
                available_groups = [g for g in groups.keys() if g != group[0]]
                random_group = random.choice(available_groups)
                groups[random_group]['Students'].extend(pair)
                # print(f'Added pair to random group {random_group} not including excluded group {group[0]}')
                added = True

        if not added:
            # Add pair to random group
            random_group = random.choice(list(groups.keys()))
            groups[random_group]['Students'].extend(pair)
            # print(f'Added pair to random group {random_group}')

    return groups


def remaining_students(students, groups, students_exclude, teachers_exclude, teachers_include):
    # Add remaining students to groups based on student exclude constraints and teacher constraints
    for student in students:
        available_groups = list(groups.keys())

        # Check if student is in teachers_include
        if student in teachers_include['Student'].values:
            teacher = teachers_include.loc[teachers_include['Student'] == student, 'Teacher'].values[0]
            teacher_group = [group for group, info in groups.items() if info['Teacher'] == teacher]
            # Add student to  that group
            groups[teacher_group[0]]['Students'].append(student)

        # Check if student is in teachers_exclude
        if student in teachers_exclude['Student'].values:
            teacher = teachers_exclude.loc[teachers_exclude['Student'] == student, 'Teacher'].values[0]
            teacher_group = [group for group, info in groups.items() if info['Teacher'] == teacher]
            available_groups = [group for group in available_groups if group != teacher_group[0]]

        # Check if student is in students_exclude
        if student in students_exclude['Student 1'].values or student in students_exclude['Student 2'].values:
            other_student = students_exclude.loc[(students_exclude['Student 1'] == student) | (students_exclude['Student 2'] == student), 'Student 1'].values[0]
            other_student_group = [group for group, info in groups.items() if other_student in info['Students']]
            if other_student_group != []:
                available_groups = [group for group in available_groups if group != other_student_group[0]]

        # Add student to a random available group
        random_group = random.choice(available_groups)
        groups[random_group]['Students'].append(student)

    return groups


def group_students(info_students, groups):
    teachers_exclude = constraints_teachers[constraints_teachers['Together'] == 'No']
    teachers_include = constraints_teachers[constraints_teachers['Together'] == 'Yes']
    students_include = constraints_students[constraints_students['Together'] == 'Yes']
    students_exclude = constraints_students[constraints_students['Together'] == 'No']
    print(teachers_exclude)

    # Set seed for reproducibility
    random.seed(seed)

    all_students = info_students['Student'].tolist()
    print(f'All students: {all_students}')

    # Make pairs based on student include constraints
    pairs = student_pairs(students_include)
    print(f'Student pairs: {pairs}')

    # Add pairs to groups based on teacher constraints
    groups = group_pairs(pairs, teachers_include, teachers_exclude)
    print(f'Groups after processing pairs with teachers: {groups}')

    # Add remaining students to groups based on constraints
    remaining_students = set(all_students) - set([student for group in groups.values() for student in group['Students']])
    groups = remaining_students(remaining_students, groups, students_exclude, teachers_exclude, teachers_include)
    print(f'Groups after adding remaining students: {groups}')

    #


g = group_students(info_students, groups)
g

Groups: {1: {'Teacher': 'T_01', 'Students': []}, 2: {'Teacher': 'T_02', 'Students': []}, 3: {'Teacher': 'T_03', 'Students': []}}
  Student Teacher Together
1    S_02    T_02       No
2    S_26    T_03       No
6    S_21    T_01       No
All students: ['S_01', 'S_02', 'S_03', 'S_04', 'S_05', 'S_06', 'S_07', 'S_08', 'S_09', 'S_10', 'S_11', 'S_12', 'S_13', 'S_14', 'S_15', 'S_16', 'S_17', 'S_18', 'S_19', 'S_20', 'S_21', 'S_22', 'S_23', 'S_24', 'S_25', 'S_26', 'S_27', 'S_28', 'S_29', 'S_30']
Student pairs: [{'S_08', 'S_21'}, {'S_29', 'S_13', 'S_10'}, {'S_03', 'S_01'}]
Groups after processing pairs with teachers: {1: {'Teacher': 'T_01', 'Students': ['S_03', 'S_01']}, 2: {'Teacher': 'T_02', 'Students': ['S_08', 'S_21']}, 3: {'Teacher': 'T_03', 'Students': ['S_29', 'S_13', 'S_10']}}


TypeError: 'set' object is not callable

In [ ]:
    # # Add remaining students to groups based on student exclude constraints and teacher constraints
    # for student in remaining_students:
    #     # Check if student is in teachers_exclude
    #     if student in teachers_exclude['Student'].values:
    #         teacher = teachers_exclude.loc[teachers_exclude['Student'] == student, 'Teacher'].values[0]
    #         group = [group for group, info in groups.items() if info['Teacher'] == teacher]
    #         available_groups = [g for g in groups.keys() if g != group[0]]

    #         # Check if student is in students_exclude
    #         if student in students_exclude['Student 1'].values or student in students_exclude['Student 2'].values:
    #             # Get the other student in the pair
    #             other_student = students_exclude.loc[(students_exclude['Student 1'] == student) | (students_exclude['Student 2'] == student), 'Student 1'].values[0]
    #             # Check if the other student is already in a group
    #             other_student_in_group = [group for group, info in groups.items() if other_student in info['Students']]
    #             print(f'Other student in group: {other_student_in_group}')
    #             if other_student_in_group:
    #                 # Add to different group
    #                 available_groups = [g for g in available_groups if g != other_student_in_group[0]]
    #                 random_group = random.choice(available_groups)
    #                 groups[random_group]['Students'].append(student)
    #                 print(f'Added student {student} to group {random_group} with other student {other_student}')
    #             else:
    #                 # Add to random group that is not the excluded group
    #                 random_group = random.choice(available_groups)
    #                 groups[random_group]['Students'].append(student)
    #                 print(f'Added student {student} to random group {random_group} with other student {other_student}')
    #         else:
    #             # Add to random group that is not the excluded group
    #             random_group = random.choice(available_groups)
    #             groups[random_group]['Students'].append(student)
    #             print(f'Added student {student} to random group {random_group}')

In [ ]:




# def cluster_students(students_include, constraints_teachers, groups):
#     # Create cluster dict with teacher and students
#     clusters = {}

#     print(groups)

#     for _, row in students_include.iterrows():
#         student_1 = row['Student 1']
#         student_2 = row['Student 2']
#         print(student_1, student_2)
#         together = row['Together']

#         matched_student = None
#         if student_1 in constraints_teachers['Student'].values:
#             matched_student = 'Student 1'
#         elif student_2 in constraints_teachers['Student'].values:
#             matched_student = 'Student 2'

#         if together == 'Yes':
#             # Check if students must be with a specific teacher
#             if matched_student:
#                 # Find the teacher for the student
#                 teacher = constraints_teachers.loc[constraints_teachers['Student'].isin([student_1, student_2]), 'Teacher'].values[0]
#                 group = [k for k, v in groups.items() if v['Teacher'] == teacher][0]

#                 # If student teacher pair is required, add them to the same group
#                 if
#                 groups[group]["Students"] = groups[group]["Students"] + [student_1, student_2]



        #     # Add both students to the same group
        #     group = random.choice(list(groups.keys()))
        #     clusters[group] = [student_1, student_2]
        # else:
        #     # Add each student to a different group
        #     group_1, group_2 = random.sample(list(groups.keys()), 2)
        #     clusters[group_1] = [student_1]
        #     clusters[group_2] = [student_2]
    # return clusters

# cluster_students(students_include, constraints_teachers, groups)

In [ ]:


groups = {}

def add_student_teacher_pairs(groups, teachers_include, info_teachers):
    # Group students by their teachers
    teachers_include = teachers_include.groupby('Teacher').agg(list).reset_index()

    # Add required student teacher pairs to groups
    for i, teacher in enumerate(info_teachers['Teacher']):
        # Check if the teacher is in the include lists
        if teacher not in teachers_include['Teacher'].values:
            students = []
        else:
            students = teachers_include.loc[teachers_include['Teacher'] == teacher, 'Student'].values[0]

        # Assign students to groups
        groups[i + 1] = {'Teacher': teacher, 'Students': students}

    return groups



def required_pairs(groups, teachers_include, info_teachers, students_include):
    together_students = students_df[students_df['Together'] == 'yes']['Student'].tolist()
    print(f'Together students: {together_students}')
    groups = add_student_teacher_pairs(groups, teachers_include, info_teachers)

    # Group students_include by student
    for student_1, student_2 in zip(students_include['Student 1'], students_include['Student 2']):
        # Check if the students are in the same group
        for group, data in groups.items():
            if student_1 in data['Students'] and student_2 not in data['Students']:
                # If they are not in the same group, add them to the same group
                groups[group]['Students'].append(student_2)
                print(f'Adding {student_2} to group {group} with {student_1}')
            elif student_2 in data['Students'] and student_1 not in data['Students']:
                # If they are not in the same group, add them to the same group
                groups[group]['Students'].append(student_1)
                print(f'Adding {student_1} to group {group} with {student_2}')

    print(f'Groups after adding required pairs: {groups}')

    # Loop through students include constraints







    return groups

required_pairs(groups, teachers_include, info_teachers, students_include)

NameError: name 'students_df' is not defined

In [ ]:
def non_required_pairs(groups, teachers_exclude):


    # print(groups)
    return groups

In [ ]:
# random baseline should:
# 1. Assign students randomly to groups
# 2. All minimum and maximum constraints are met

# 3. Ensure that students with yes constraints are in the same group
# 4. Ensure that students with no constraints are not in the same group
# 5. Ensure that students and teacher with yes constraints are in the same group
# 6. Ensure that students and teacher with no constraints are not in the same group







def random_baseline(n_groups, seed=42):
    # Set seed for reproducibility
    random.seed(seed)

    # Initialize groups
    groups = {i: [] for i in range(1, n_groups + 1)}
    teacher_groups = {teacher: i + 1 for i, teacher in enumerate(info_teachers['Teacher'])}
    print("groups: ", groups)
    print("teacher_groups: ", teacher_groups)

    # Split constraints
    students_include = constraints_students[constraints_students['Together'] == 'Yes']
    students_exclude = constraints_students[constraints_students['Together'] == 'No']
    teachers_include = constraints_teachers[constraints_teachers['Together'] == 'Yes']
    teachers_exclude = constraints_teachers[constraints_teachers['Together'] == 'No']
    print(teachers_include)

    # Create a list of students
    students = list(info_students['Student'])

    # Add teacher include constraints first
    for _, row in teachers_include.iterrows():
        print(row)
        student = row['Student']
        teacher = row['Teacher']

        # Get the group assigned to this teacher and add the student to that group
        group_number = teacher_groups.get(teacher)

        if group_number:
            groups[group_number].append(student)

    # Print groups to see the result
    print("Groups after assigning students based on teacher:")
    print(groups)



random_baseline(n_groups, seed=42)


groups:  {1: [], 2: [], 3: []}
teacher_groups:  {'T_01': 1, 'T_02': 2, 'T_03': 3, 'T_04': 4, 'T_05': 5}
  Student Teacher Together
0    S_10    T_03      Yes
3    S_28    T_05      Yes
4    S_13    T_02      Yes
5    S_02    T_03      Yes
6    S_30    T_03      Yes
Student     S_10
Teacher     T_03
Together     Yes
Name: 0, dtype: object
Student     S_28
Teacher     T_05
Together     Yes
Name: 3, dtype: object


KeyError: 5

In [ ]:
def build_clusters_with_teacher_constraints(constraints_students, constraints_teachers, n_groups):
    from collections import defaultdict

    # Students that should be paired
    student_pairs = list(constraints_students[constraints_students['Together'] == 'Yes'][['Student 1', 'Student 2']].itertuples(index=False))
    print("student_pairs: ", student_pairs)


    clusters = []
    for a, b in student_pairs:
        found = False
        for cluster in clusters:
            if a in cluster or b in cluster:
                cluster.update([a, b])
                found = True
                break
        if not found:
            clusters.append(set([a, b]))

    # Merge overlapping clusters
    merged = []
    while clusters:
        base, *rest = clusters
        base = set(base)
        for other in rest[:]:
            if base & other:
                base |= other
                rest.remove(other)
        merged.append(base)
        clusters = rest

    # Step 2: Add unclustered students
    all_students = set(info_students['Student'])
    clustered = set().union(*merged) if merged else set()
    for student in all_students - clustered:
        merged.append(set([student]))

    # Step 3: Map students to clusters
    student_to_cluster = {}
    for i, cluster in enumerate(merged):
        for student in cluster:
            student_to_cluster[student] = i

    # Step 4: Attach teachers to clusters
    cluster_teachers = defaultdict(set)
    for _, row in constraints_teachers[constraints_teachers['Together'] == 'Yes'].iterrows():
        student, teacher = row['Student'], row['Teacher']
        cluster_id = student_to_cluster.get(student)
        if cluster_id is not None:
            cluster_teachers[cluster_id].add(teacher)

    # Step 5: Bundle clusters with their teachers
    full_clusters = [(merged[i], cluster_teachers.get(i, set())) for i in range(len(merged))]

    # Step 6: If too many clusters, merge small ones greedily
    while len(full_clusters) > n_groups:
        # Sort by cluster size (smallest first)
        full_clusters.sort(key=lambda x: len(x[0]))

        merged = False
        for i in range(len(full_clusters)):
            for j in range(i + 1, len(full_clusters)):
                c1, t1 = full_clusters[i]
                c2, t2 = full_clusters[j]
                if not t1 & t2:  # no teacher conflict
                    new_cluster = (c1 | c2, t1 | t2)
                    full_clusters.pop(j)
                    full_clusters.pop(i)
                    full_clusters.append(new_cluster)
                    merged = True
                    break
            if merged:
                break
        if not merged:
            raise ValueError("Cannot reduce clusters to required number without violating teacher constraints.")

    return full_clusters

build_clusters_with_teacher_constraints(constraints_students, constraints_teachers, n_groups)

student_pairs:  [Pandas(_0='S_08', _1='S_21'), Pandas(_0='S_29', _1='S_10')]


[({'S_01', 'S_07', 'S_09', 'S_17', 'S_18', 'S_19', 'S_24', 'S_30'}, set()),
 ({'S_02', 'S_06', 'S_11', 'S_13', 'S_16', 'S_20', 'S_23', 'S_26'}, set()),
 ({'S_03',
   'S_04',
   'S_05',
   'S_08',
   'S_10',
   'S_12',
   'S_14',
   'S_15',
   'S_21',
   'S_22',
   'S_25',
   'S_27',
   'S_28',
   'S_29'},
  {'T_03', 'T_05'})]

In [ ]:






def assign_students_randomly(student_info, constraints, num_groups=3, seed=42):
    # Set seed for reproducibility
    random.seed(seed)

    groups = {i: [] for i in range(num_groups)}
    include_constraints = {}
    exclude_constraints = {}

    for _, row in constraints.iterrows():
        s1, s2 = row['Student_1'], row['Student_2']
        if row['Constraint'] == 'Include':
            include_constraints.setdefault(s1, []).append(s2)
            include_constraints.setdefault(s2, []).append(s1)
        elif row['Constraint'] == 'Exclude':
            exclude_constraints.setdefault(s1, []).append(s2)
            exclude_constraints.setdefault(s2, []).append(s1)

    # Shuffle students for randomness
    students = student_info.sample(frac=1, random_state=seed).reset_index(drop=True)


    assigned = {}
    # Assign students while considering constraints
    for _, student in students.iterrows():
        student_id = student['ID']
        possible_groups = list(groups.keys())

        # Ensure no exclusions
        for group_id in possible_groups[:]:
            if any(ex in groups[group_id] for ex in exclude_constraints.get(student_id, [])):
                possible_groups.remove(group_id)

        # Assign to least populated valid group
        chosen_group = min(possible_groups, key=lambda g: len(groups[g]))
        groups[chosen_group].append(student_id)
        assigned[student_id] = chosen_group

        # Ensure inclusions
        for inc in include_constraints.get(student_id, []):
            if inc in assigned:
                continue
            groups[chosen_group].append(inc)
            assigned[inc] = chosen_group

    result = pd.DataFrame([(s, g) for g, students in groups.items() for s in students],
                          columns=['ID', 'Assigned_Group'])

    return result


assigned_groups = assign_students_randomly(student_info, constraints, seed=48)
print(assigned_groups)

# Save the assigned groups to a CSV file
assigned_groups.to_csv('assigned_groups.csv', index=False)

NameError: name 'student_info' is not defined